In [1]:
pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 2.8 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [1]:
import duckdb
print(duckdb.__version__)

1.5.6


In [2]:
con = duckdb.connect("/home/jovyan/work/analytics.duckdb")
print(con.execute("SELECT version()").fetchone())

('v1.5.6',)


In [4]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

HTTPException: HTTP Error: Failed to download extension "httpfs" at URL "http://extensions.duckdb.org/v1.5.6/linux_amd64/httpfs.duckdb_extension.gz" (HTTP 302)
Extension "httpfs" is an existing extension.

For more info, visit https://duckdb.org/docs/stable/extensions/troubleshooting?version=v1.5.6&platform=linux_amd64&extension=httpfs

In [4]:
con.execute("""
SET s3_endpoint='minio:9000';
SET s3_access_key_id='minio';
SET s3_secret_access_key='minio123';
SET s3_use_ssl=false;
SET s3_url_style='path';
""")

In [5]:
tables = {
    "dim_users": "s3a://end-to-end-streaming-data-platform-gold/dim_users",
    "dim_videos": "s3a://end-to-end-streaming-data-platform-gold/dim_videos",
    "fact_subscription": "s3a://end-to-end-streaming-data-platform-gold/fact_subscription",
    "fact_payments": "s3a://end-to-end-streaming-data-platform-gold/fact_payments",
    "fact_events": "s3a://end-to-end-streaming-data-platform-gold/fact_events",
}

for table_name, parquet_path in tables.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_parquet('{parquet_path}/*.parquet')
    """)

    print(f"Loaded {table_name}")

Loaded dim_users
Loaded dim_videos
Loaded fact_subscription
Loaded fact_payments
Loaded fact_events


In [5]:
tables = con.execute("SHOW TABLES").fetchall()

for table in tables:
    print(table[0])

dim_users
dim_videos
fact_events
fact_payments
fact_subscription


In [6]:
print(con.execute("SELECT current_database()").fetchone())
print(con.execute("SHOW TABLES").fetchall())

('analytics',)
[('dim_users',), ('dim_videos',), ('fact_events',), ('fact_payments',), ('fact_subscription',)]


In [8]:
print(con.execute("PRAGMA database_list").fetchall())

[(20624, 'analytics', '/home/jovyan/work/analytics.duckdb')]


In [9]:
print(con.execute("SHOW TABLES").fetchall())

[('dim_users',), ('dim_videos',), ('fact_events',), ('fact_payments',), ('fact_subscription',)]


In [6]:
for table_name, in con.execute("SHOW TABLES").fetchall():
    print(f"\n--- {table_name} ---")
    print(con.execute(
        f"DESCRIBE {table_name}"
    ).fetchall())


--- dim_users ---
[('user_id', 'VARCHAR', 'YES', None, None, None), ('email', 'VARCHAR', 'YES', None, None, None), ('country', 'VARCHAR', 'YES', None, None, None), ('account_status', 'VARCHAR', 'YES', None, None, None), ('created_at', 'DATE', 'YES', None, None, None), ('last_login', 'DATE', 'YES', None, None, None)]

--- dim_videos ---
[('video_id', 'VARCHAR', 'YES', None, None, None), ('match_id', 'VARCHAR', 'YES', None, None, None), ('content_type', 'VARCHAR', 'YES', None, None, None), ('status', 'VARCHAR', 'YES', None, None, None), ('title', 'VARCHAR', 'YES', None, None, None), ('duration_sec', 'INTEGER', 'YES', None, None, None), ('competition_name', 'VARCHAR', 'YES', None, None, None), ('technical_max_resolution', 'VARCHAR', 'YES', None, None, None), ('stats_views', 'INTEGER', 'YES', None, None, None), ('stats_likes', 'INTEGER', 'YES', None, None, None), ('language_1', 'VARCHAR', 'YES', None, None, None), ('language_2', 'VARCHAR', 'YES', None, None, None), ('created_at', 'DATE', 

In [6]:
tables = {
    "dim_users": "user_id",
    "dim_videos": "video_id",
    "fact_events": "event_id",
    "fact_payments": "payment_id",
    "fact_subscription": "subscription_id",
}

for table, key in tables.items():
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE {key} IS NULL
            ) AS null_keys,
            COUNT(DISTINCT {key}) AS unique_keys
        FROM {table}
    """).fetchone()

    print(
        f"{table}: "
        f"rows={result[0]}, "
        f"null_keys={result[1]}, "
        f"unique_keys={result[2]}"
    )

dim_users: rows=9997, null_keys=0, unique_keys=9997
dim_videos: rows=6235, null_keys=0, unique_keys=6235
fact_events: rows=8796, null_keys=0, unique_keys=8796
fact_payments: rows=11767, null_keys=0, unique_keys=11767
fact_subscription: rows=20140, null_keys=0, unique_keys=20140


In [9]:
query = """
SELECT
    'events -> users' AS relationship,
    COUNT(*) AS unmatched_rows
FROM fact_events e
LEFT JOIN dim_users u ON e.user_id = u.user_id
WHERE u.user_id IS NULL

UNION ALL

SELECT
    'events -> videos',
    COUNT(*)
FROM fact_events e
LEFT JOIN dim_videos v ON e.video_id = v.video_id
WHERE v.video_id IS NULL

UNION ALL

SELECT
    'subscriptions -> users',
    COUNT(*)
FROM fact_subscription s
LEFT JOIN dim_users u ON s.user_id = u.user_id
WHERE u.user_id IS NULL

UNION ALL

SELECT
    'payments -> subscriptions',
    COUNT(*)
FROM fact_payments p
LEFT JOIN fact_subscription s
    ON p.subscription_id = s.subscription_id
WHERE s.subscription_id IS NULL
"""

print(con.execute(query).fetchall())

[('events -> users', 5), ('events -> videos', 4763), ('subscriptions -> users', 6), ('payments -> subscriptions', 1)]


In [10]:
query = """
SELECT
    e.video_id,
    e.interaction_type,
    COUNT(*) AS event_count
FROM fact_events e
LEFT JOIN dim_videos v
    ON e.video_id = v.video_id
WHERE v.video_id IS NULL
GROUP BY e.video_id, e.interaction_type
ORDER BY event_count DESC
LIMIT 20
"""

print(con.execute(query).fetchall())

[('01a0f05a-e65d-7fba-ad00-428f32ce2513', 'pause', 11), ('01a0f05a-fefa-7bea-aead-8cf5379bd282', 'like', 10), ('01a0f05a-ee30-75ca-9c00-981b7bba5166', 'pause', 9), ('01a0f05a-effb-73db-a223-0ae07175f3b3', 'pause', 8), ('01a0f05a-eed9-7f3e-b045-560d6c48fd4b', 'pause', 8), ('01a0f05a-e580-77de-91af-892d4b9034b8', 'like', 8), ('01a0f05a-f79c-78fd-8de2-15e29cb6a9e8', 'pause', 8), ('01a0f05b-00af-7dec-8ae5-f0fb83c3db58', 'like', 8), ('01a0f05a-f30e-75ca-acd4-f30d68d0f3eb', 'pause', 7), ('01a0f05a-edd2-7df2-ab5d-96f5a1a03571', 'like', 7), ('01a0f05a-f58c-7346-b97b-c37ddd24f2bc', 'pause', 7), ('01a0f05a-fad3-7fed-a558-32855b6c9c90', 'like', 7), ('01a0f05a-f93e-767a-b2ad-a549720328eb', 'pause', 7), ('01a0f05a-e88c-7d28-bac7-188390861e28', 'like', 7), ('01a0f05a-f507-7f82-8792-d055c2e6a8da', 'like', 7), ('01a0f05a-f012-7308-91a7-64f138230f08', 'like', 7), ('01a0f05a-ffd1-7254-8972-8c74eae66c29', 'like', 7), ('01a0f05b-01d5-7f39-8ba2-d28a6485aab1', 'like', 7), ('01a0f05a-dfe2-72fa-88e8-788e9313f

In [3]:
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN (
    'dim_users',
    'dim_videos',
    'fact_events',
    'fact_subscriptions',
    'fact_payments'
)
ORDER BY table_name, ordinal_position;

SyntaxError: invalid syntax (2031164138.py, line 1)

In [4]:
query = """
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name IN (
    'dim_users',
    'dim_videos',
    'fact_events',
    'fact_subscriptions',
    'fact_payments'
)
ORDER BY table_name, ordinal_position
"""

con.sql(query).show()

┌───────────────┬──────────────────┬───────────────┐
│  table_name   │   column_name    │   data_type   │
│    varchar    │     varchar      │    varchar    │
├───────────────┼──────────────────┼───────────────┤
│ dim_users     │ user_id          │ VARCHAR       │
│ dim_users     │ email            │ VARCHAR       │
│ dim_users     │ country          │ VARCHAR       │
│ dim_users     │ account_status   │ VARCHAR       │
│ dim_users     │ created_at       │ DATE          │
│ dim_users     │ last_login       │ DATE          │
│ dim_videos    │ video_id         │ VARCHAR       │
│ dim_videos    │ match_id         │ VARCHAR       │
│ dim_videos    │ content_type     │ VARCHAR       │
│ dim_videos    │ status           │ VARCHAR       │
│     ·         │   ·              │    ·          │
│     ·         │   ·              │    ·          │
│     ·         │   ·              │    ·          │
│ fact_events   │ video_id         │ VARCHAR       │
│ fact_events   │ interaction_type │ VARCHAR  

In [8]:
query = """
SELECT
    'fact_events -> dim_users' AS relationship,
    COUNT(*) AS unmatched_rows
FROM fact_events f
LEFT JOIN dim_users d ON f.user_id = d.user_id
WHERE d.user_id IS NULL

UNION ALL

SELECT
    'fact_events -> dim_videos',
    COUNT(*)
FROM fact_events f
LEFT JOIN dim_videos d ON f.video_id = d.video_id
WHERE d.video_id IS NULL

UNION ALL

SELECT
    'fact_subscription -> dim_users',
    COUNT(*)
FROM fact_subscription f
LEFT JOIN dim_users d ON f.user_id = d.user_id
WHERE d.user_id IS NULL

UNION ALL

SELECT
    'fact_payments -> fact_subscription',
    COUNT(*)
FROM fact_payments p
LEFT JOIN fact_subscription s
    ON p.subscription_id = s.subscription_id
WHERE s.subscription_id IS NULL
"""

con.sql(query).show()

┌────────────────────────────────────┬────────────────┐
│            relationship            │ unmatched_rows │
│              varchar               │     int64      │
├────────────────────────────────────┼────────────────┤
│ fact_events -> dim_users           │              5 │
│ fact_events -> dim_videos          │           4763 │
│ fact_subscription -> dim_users     │              6 │
│ fact_payments -> fact_subscription │              1 │
└────────────────────────────────────┴────────────────┘



In [9]:
query = """
SELECT 'dim_users' AS table_name,
       COUNT(*) AS total_rows,
       COUNT(user_id) AS non_null_keys,
       COUNT(DISTINCT user_id) AS unique_keys
FROM dim_users

UNION ALL

SELECT 'dim_videos',
       COUNT(*),
       COUNT(video_id),
       COUNT(DISTINCT video_id)
FROM dim_videos

UNION ALL

SELECT 'fact_events',
       COUNT(*),
       COUNT(event_id),
       COUNT(DISTINCT event_id)
FROM fact_events

UNION ALL

SELECT 'fact_subscription',
       COUNT(*),
       COUNT(subscription_id),
       COUNT(DISTINCT subscription_id)
FROM fact_subscription

UNION ALL

SELECT 'fact_payments',
       COUNT(*),
       COUNT(payment_id),
       COUNT(DISTINCT payment_id)
FROM fact_payments
"""

con.sql(query).show()

┌───────────────────┬────────────┬───────────────┬─────────────┐
│    table_name     │ total_rows │ non_null_keys │ unique_keys │
│      varchar      │   int64    │     int64     │    int64    │
├───────────────────┼────────────┼───────────────┼─────────────┤
│ dim_users         │       9997 │          9997 │        9997 │
│ dim_videos        │       6235 │          6235 │        6235 │
│ fact_events       │       8796 │          8796 │        8796 │
│ fact_subscription │      20140 │         20140 │       20140 │
│ fact_payments     │      11767 │         11767 │       11767 │
└───────────────────┴────────────┴───────────────┴─────────────┘



In [10]:
con.close()